# Graded Lab 1: Reflection

**CSI 4900 / CSI 5900 — Special Topics: Agentic AI**  ·  Oakland University

---

When you ask a language model for something, it answers in one shot. It is
writing the answer while it is still deciding what the answer is, and it never
looks at the finished product.

**Reflection** is the fix, and it is the simplest idea in this course. Instead of
one call to the model, you make three:

1. Ask it to write something.
2. Ask it what is wrong with what it just wrote.
3. Ask it to rewrite, using that list of problems.

That is the whole pattern. This lab has you build it.

## How this notebook is organised

| Part | What it is | You write code? | Graded? |
|---|---|---|---|
| **A** | A worked example — flashcards that get better. Read it, run it. | No | No |
| **B** | The assignment: three functions you write | **Yes** | **Yes — all 100 points** |
| **C** | Your three functions, run once on a real abstract | No | No |
| **D** | The same thing over five abstracts, with the scores compared | No | No |

Part A shows you the pattern before you build it. Parts C and D are where you
watch the thing you built actually work.

## What Part B asks for

A research abstract has to be rewritten into plain English, and the result has to
obey four rules: a word limit, two facts it must state exactly, a list of words it
may not use, and a limit on sentence length.

You will write three functions — one to write the draft, one to criticise it, one
to fix it. Each is a single prompt and a single call to the model.

Here is what makes this more than "write a better prompt". Your critic is handed
the output of a **rule checker**: a plain Python function that tells it exactly
which rules the draft broke. A critic working from facts gives far more useful
instructions than a critic working from impressions, and in Part D you will see
the difference come out as a number.

## Grading (100 points)

| | |
|---|---|
| Exercise 1 — `generate_summary` | 30 |
| Exercise 2 — `critique_summary` | 40 |
| Exercise 3 — `revise_summary` | 30 |

All three are graded by `unittests.py`, which runs **without the internet and
without your API key**. Run it as often as you want; it can never use up your free
quota. Parts C and D are not graded at all.

## Three rules

* **Do not edit `utils.py` or `unittests.py`.**
* **Do not write the summaries yourself.** Every function has to get its text from
  the model. A summary you typed by hand scores zero even if it is perfect, and
  the tests can tell the difference.
* **The prompts are the assignment.** Rewriting a prompt until it works is the
  skill this lab teaches, so expect to do it more than once.

## How long this should take

About 45 minutes to an hour: roughly 10 minutes reading and running Part A,
25 minutes on the three exercises, 10 minutes running Parts C and D.

## Setup

You need a **free** Gemini API key. It takes about three minutes and does not
require a credit card.

1. Go to <https://aistudio.google.com/apikey> and sign in with any Google account.
2. Click **Create API key**, then copy the key.
3. In the folder that holds this notebook, make a copy of `.env.example` and name
   the copy `.env`.
4. Open `.env`, paste your key after `GEMINI_API_KEY=`, and save. No spaces, no
   quotation marks around the key.

Install the dependencies once, from that same folder:

```bash
pip install -r requirements.txt
```

Then run the cell below. If it prints `client ready`, everything works.

**If it does not**, the cause is almost always one of two things: the notebook is
running a different Python than the one you installed into, or `.env` is not where
the notebook is looking for it. `README.md` in this folder lists the exact error
messages and what to do about each one.

Your API key is personal. Do not paste it into a notebook cell, do not upload
`.env` anywhere, and do not put anything confidential into a prompt — Google
states that free-tier traffic may be used to improve their products.

In [ ]:
import utils
from utils import (
    BENCHMARK_TASKS,
    CRITIC_MODEL,
    DRAFT_MODEL,
    PRACTICE_TASK,
    STUDY_PASSAGE,
    check_summary,
    get_client,
    show_output,
)

client = get_client()

print("client ready")
print("drafting model:", DRAFT_MODEL)
print("critic model:  ", CRITIC_MODEL)

client ready
drafting model: openai:gemini-3.5-flash-lite
critic model:   openai:gemini-3.5-flash-lite


---

# Part A — A worked example: flashcards that get better

*Read this and run it. None of it is graded, and you write no code here.*

Someone pastes a paragraph from a textbook into a chatbot and asks for study
flashcards. They get three flashcards back. Are they *good* flashcards? That is a
separate question — and it turns out the model can answer it, if somebody asks.

We do it in three calls to the model:

1. **Generate** — write three flashcards from the passage.
2. **Critique** — judge those flashcards against three rules about what makes a
   flashcard useful.
3. **Revise** — rewrite them, using the critique.

**Read the three functions carefully.** Part B asks you to write the same three
functions for a different task, so this is your reference. When you get stuck
later, scroll back up here.

One thing to watch for as you read: **the three calls know nothing about each
other.** The model has no memory between them. When the critic sees the
flashcards, it is only because we copied them into its prompt ourselves with an
f-string — and the same goes for the reviser. Moving results from one step into
the next by hand is the entire trick. There is nothing else going on.

In [2]:
print(STUDY_PASSAGE)

A model that is trained for too long will eventually start to fit the noise in
its training set rather than the pattern underneath it. The symptom is easy to
see: training error keeps falling while error on held-out data starts to rise.
Regularization is the family of techniques used to push back on this. Weight
decay adds a penalty proportional to the size of the weights, which keeps any
single feature from dominating. Dropout randomly switches off units during
training, so the network cannot lean on one pathway. Early stopping simply ends
training at the point where held-out error is lowest. None of these techniques
add information; they all trade a little training accuracy for a model that
generalizes better.


### Step 1 — Generate

A plain request, with nothing said about quality. This is the "one-shot" answer —
the thing reflection has to improve on.

Look at the shape of the function below, because every function you write in
Part B has these same three steps:

1. **Build a prompt** — an f-string holding everything the model needs to see.
2. **Send it** — one call to `client.chat.completions.create(...)`.
3. **Return what came back** — `response.choices[0].message.content`.

In [3]:
def make_flashcards(passage, client, model=DRAFT_MODEL):
    # 1. build the prompt (an f-string: everything the model needs goes in here)
    prompt = f"""Write exactly three study flashcards from the passage below.
Format each one as:

Q: <question>
A: <answer>

PASSAGE:
{passage}"""

    # 2. send it
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )

    # 3. return the model's reply
    return response.choices[0].message.content


cards_v1 = make_flashcards(STUDY_PASSAGE, client)
show_output("Step 1 — first draft", cards_v1, background="#fff8dc")

### Step 2 — Critique

A new call with a new job: we hand the model the draft plus a short standard to
judge it against, and we tell it explicitly **not to fix anything**.

Why keep criticising and fixing apart? Two reasons.

* A model asked to do both at once tends to quietly fix the easy problems and
  skip the hard ones — and from the result alone you cannot tell which is which.
* When they are separate, the critique is a piece of text you can read. If the
  final answer comes out bad, the critique tells you whether the agent failed to
  *notice* the flaw or failed to *fix* it. Those need different repairs.

Notice also that this call passes `CRITIC_MODEL` while Step 1 passed
`DRAFT_MODEL`. In a system you were paying for, those would be two *different*
models — judging work is harder than producing it, so the critic would get the
stronger and more expensive one.

On the free tier we only get one usable class of model, so right now both
constants point at the same one. They are kept separate anyway, because that
makes upgrading the critic a one-line change in `utils.py` instead of a hunt
through every prompt. Your Part B functions take the same two arguments for the
same reason.

In [4]:
def critique_flashcards(cards, client, model=CRITIC_MODEL):
    prompt = f"""You are reviewing study flashcards. A good flashcard:

  1. asks a question that can be answered without having the passage in front of you,
  2. has an answer that does not simply repeat the words of the question,
  3. tests exactly one idea, not three at once.

Go through the cards below one at a time and say which of these three
requirements each card fails, and why. Be specific and be blunt.
Do NOT rewrite the cards.

CARDS:
{cards}"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content


critique = critique_flashcards(cards_v1, client)
show_output("Step 2 — critique", critique, background="#e0f7fa")

### Step 3 — Revise

The reviser is given two things: the original draft **and** the critique. Both are
pasted into the prompt.

That is worth pausing on. The draft was written by a model, but *this* call has no
memory of writing it. If we did not paste the draft back in, there would be
nothing here to revise.

In [5]:
def revise_flashcards(cards, critique, client, model=DRAFT_MODEL):
    prompt = f"""Rewrite the flashcards below so that every problem raised by the
reviewer is fixed. Keep the same Q:/A: format and keep exactly three cards.
Return only the cards.

CARDS:
{cards}

REVIEWER'S NOTES:
{critique}"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content


cards_v2 = revise_flashcards(cards_v1, critique, client)
show_output("Step 3 — revision", cards_v2, background="#f3e5f5")

### What just happened — and what is missing

Compare `cards_v1` with `cards_v2`. The second version is usually tighter. Nothing
new entered the system: same model, same passage, same information. The only thing
that changed is that we made it look at its own work before rewriting.

Now pick one card that changed and ask a harder question: **is it actually
better?**

Here is a pattern you will see a lot. The critic says "this card tests two ideas
at once." The reviser obeys — by *deleting one of the two ideas*. The rule is now
satisfied. Real content is now gone. And nothing anywhere in this pipeline can
tell those two outcomes apart: the critic is satisfied, the reviser did as it was
told, and at no point did anyone measure anything.

That is the limit of reflection on its own. **The only judge is the model's
opinion of its own work**, and models are agreeable about that.

Part B fixes this by giving the critic a tool: a plain Python function that reports
facts instead of opinions.

---

# Part B — The assignment

## The task

A research group wants plain-English summaries of its papers for the department's
public newsletter. Every summary has to satisfy four rules:

1. **Length** — 100 words or fewer.
2. **Grounding** — two specific pieces of text must appear exactly as given: the
   name of the dataset, and the headline number. Writing "about 89" instead of
   `88.6`, or getting the dataset's name slightly wrong, breaks this rule. It is
   the most important of the four — a summary that reads beautifully and
   misreports the result is worse than no summary at all.
3. **Plain language** — certain technical words are banned outright. Related forms
   of a banned word are banned too: if `distillation` is banned, so are
   `distillations` and `distillation's`.
4. **Readability** — no sentence may be longer than 20 words.

A function called `check_summary()`, already written for you in `utils.py`, checks
all four and scores the summary out of 4. It is ordinary Python — counting words
and matching strings — so it is never wrong and it costs nothing to run.

Run the next two cells to see the task and then the checker.

In [ ]:
task = PRACTICE_TASK  # the practice task; the graded benchmark uses five others

print("TITLE:", task.title)
print()
print("ABSTRACT:")
print(task.abstract)
print()
print("RULES:")
print(task.rules_text())

TITLE: Cold-Start Recommendation with Session Signals

ABSTRACT:
Recommenders degrade badly for accounts with no interaction history. We propose a two-tower architecture in which the user tower is conditioned on within-session behavioural signals - dwell time, scroll depth, and query reformulations - rather than on a learned user embedding. This removes the dependence on historical interactions entirely for new accounts, at the cost of a slightly weaker personalization signal for long-tenured ones. On the ShopStream-4M click log our approach improves recall@20 for first-session users from 0.121 to 0.194, a 60 percent relative gain, while leaving performance on established accounts unchanged within noise. We also report an online A/B test over three weeks confirming the offline result.

RULES:
1. At most 100 words in total.
2. Must contain these exact strings, spelled exactly as given: 'ShopStream-4M', '0.194'
3. Must NOT use any of these words or phrases (or forms of them): 'embedding'

Below is a draft that reads perfectly well and breaks three of the four rules.
Watch what the checker says about it.

**Look closely at the shape of this report.** In Exercise 2 your critic prompt has
to contain it, because it is what turns a vague critic into a specific one.

To be clear about what that means: you do **not** copy the text you see printed
below into your code. You call `check_summary()` on whatever draft your function
was given, and drop the string it returns into the f-string, the same way Part A
dropped `{passage}` and `{cards}` into its prompts.

In [ ]:
plausible_but_wrong = (
    "The authors describe a two-tower architecture in which the user embedding is "
    "replaced by within-session behavioural signal features such as dwell time, "
    "scroll depth and query reformulations, which lets the system produce a useful "
    "personalization signal for accounts with no interaction history, and they "
    "report a substantial improvement in recall on the ShopStream-4M click log."
)

result = check_summary(plausible_but_wrong, task)
print(result["report"])
print()
print(
    "passed:", result["passed"], "| score:", result["score"], "/", result["max_score"]
)

RULE CHECK REPORT
score: 1/4
  [PASS] length: 55 words (limit 100)
  [FAIL] grounding: missing: '0.194'
  [FAIL] plain_language: banned terms used: 'embedding', 'two-tower', 'recall', 'architecture', 'signal', 'personalization'
  [FAIL] readability: 55 words: "The authors describe a two-tower architecture in which the u..."

passed: False | score: 1 / 4


## What you are building

```
generate_summary(task)                 ->  a draft summary
        |
        v
   check_summary(draft, task)          ->  a report on which rules it broke
        |                                  (already written for you)
        v
critique_summary(draft, task)          ->  instructions for fixing it
        |                                  (your critic must use that report)
        v
revise_summary(draft, critique, task)  ->  a better draft
```

Three functions, three exercises. **Each one is a single call to the model**, with
the same three steps as `make_flashcards` in Part A: build a prompt, send it,
return the reply. Scroll back to that function whenever you lose the shape.

## About the `client` argument

Each stub below starts with this line, already written for you:

```python
client = client or get_client()
```

**Leave it exactly as it is.** It reads: if somebody handed this function a client,
use that one; otherwise build the normal one.

The "somebody" is the test suite. When the tests run your function, they pass in a
stand-in that never contacts Google — it just records the prompt you sent it and
hands back a fixed string. That is how the tests can check your work with no
internet, no API key, and no cost to your quota. Beyond keeping the line, you do
not need to do anything about it.

---

### Exercise 1 — `generate_summary`  *(30 points)*

Write the first draft.

**What the function does**: it is given a task, asks the model for a plain-English
summary of `task.abstract`, and returns the model's reply as a string.

**What the tests check** (offline, no API calls):

* your function makes exactly **one** call to `client.chat.completions.create(...)`
* the prompt you send contains `task.abstract`
* the prompt you send also states the rules — both the required strings and the
  banned words
* your function returns the model's reply **word for word**, with nothing added,
  removed or edited by your Python code

**The shortcut worth knowing**: `task.rules_text()` returns all four rules already
written out in English. Put the *call* inside your f-string and Python will
substitute the text for you at run time:

```python
prompt = f"""...

RULES:
{task.rules_text()}
"""
```

That is the easiest way to satisfy the third bullet above. Run
`print(task.rules_text())` in any cell if you want to see what it produces — but
do not copy that output into your code. Let the f-string do it, so the rules
always match whichever task your function is handed.

**What `task` gives you**

| | |
|---|---|
| `task.title` | the paper's title |
| `task.abstract` | the text to be rewritten |
| `task.rules_text()` | all four rules as English text, ready to paste |
| `task.must_include` | the list of strings that must appear |
| `task.banned_terms` | the list of words that must not appear |
| `task.max_words` | 100 |
| `task.max_sentence_words` | 20 |

In [8]:
def generate_summary(task, client=None, model=DRAFT_MODEL):
    """Write a first-draft plain-language summary of `task`."""
    client = client or get_client()

    ### START CODE HERE ###

    # Step 1 - build the prompt, exactly like make_flashcards did in Part A.
    #   Use an f-string. It MUST contain all three of these, or the tests fail:
    #       {task.title}          - what the paper is called
    #       {task.abstract}       - the text to rewrite
    #       {task.rules_text()}   - all four rules, already written out for you
    #                               (put the call inside the f-string, exactly
    #                                like that - do not paste its output)
    #   Also tell the model who it is writing for, and to return ONLY the summary
    #   (no heading, no "Here is the summary:" - a preamble like that counts
    #   toward the 100-word limit and can break the length rule on its own).
    prompt = f"""Write a plain-English summary for the following abstract from the paper titled "{task.title}". The summary is intended for the department's public newsletter.
Abstract: {task.abstract}
Rules: {task.rules_text()}
Return only the summary, no headings or preamble."""

    # Step 2 - send it. Copy this call from make_flashcards:
    #       client.chat.completions.create(
    #           model=model,
    #           messages=[{"role": "user", "content": prompt}],
    #           temperature=0.7,
    #       )
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )

    # Step 3 - return the model's reply and nothing else.
    #   It lives at  response.choices[0].message.content
    #   Add .strip() to drop stray blank lines.
    return response.choices[0].message.content.strip()

    ### END CODE HERE ###

Run the test. It uses the stand-in model described above, so it costs nothing and
you can run it as often as you like.

If it fails, read the message. It names the check that did not pass and says what
to do about it.

In [9]:
import unittests

print(unittests.test_generate_summary(generate_summary)[1])

PASS  test_generate_summary


---

### Exercise 2 — `critique_summary`  *(40 points)*

This is the exercise the lab is really about, and it is worth the most points.

In Part A the critic worked from nothing but its own impressions. Here it gets a
tool. Your function runs `check_summary()` on the draft **first**, then hands the
model that report and asks it to turn those facts into instructions a writer can
follow.

The difference is dramatic in practice:

* A critic with no report says: *"consider making the language more accessible."*
* A critic holding the report says: *"`0.194` is missing — add it. Replace the word
  `signal`. Split the second sentence; it is 26 words long."*

The second one can be acted on. The first one cannot.

**What the function does**: it is given a draft summary and a task, and returns
written instructions for fixing that draft.

**What the tests check**:

* your function calls `check_summary(summary, task)` and puts its `["report"]`
  into the prompt — the test looks for that report's text in what you sent
* the prompt contains the draft being critiqued
* exactly **one** call to the model
* the function returns the model's reply word for word

The test runs your function twice, on two different drafts, and checks that the
report in your prompt changes to match. So the checker really has to run **inside
your function, on the `summary` it was given** — a report you copied into the code
by hand will fail on the second draft.

**Two things your prompt has to say**

1. That the checker is never wrong about the rules, so every `FAIL` line is a fact
   to act on rather than an opinion to weigh up.
2. That the model must **not rewrite the summary** — instructions only. If it
   hands back a finished rewrite, Exercise 3 has nothing left to do, and you have
   merged two steps into one. Step 2 of Part A explains why that goes badly.

In [10]:
def critique_summary(summary, task, client=None, model=CRITIC_MODEL):
    """Turn the rule checker's verdict into instructions a writer can act on."""
    client = client or get_client()

    ### START CODE HERE ###

    # Step 1 - run the checker on the draft and pull out its report.
    #   check_summary(summary, task) returns a dict. Index it with ["report"]
    #   to get the printable text out of it - that is one string.
    #   Scroll up to the cell that printed "RULE CHECK REPORT" to see what that
    #   string looks like. Do NOT copy that text into your code: call the
    #   checker here, on the `summary` this function was given.
    result = check_summary(summary, task)
    # print(result["report"])
    # print()
    # print("passed:", result["passed"], "| score:", result["score"], "/", result["max_score"])

    # Step 2 - build the prompt. It MUST contain:
    #       {summary}             - the draft being critiqued
    #       {report}              - the checker's output   <- the tests look for this
    #       {task.rules_text()}   - the rules, for context
    #   Tell the model to treat every FAIL line as fact, to give concrete
    #   instructions (which words to cut, what to add, which sentences to split),
    #   and NOT to rewrite the summary itself.
    prompt = f"""Critique the given summary based on the following report and rules. Treat every FAIL line as fact. Give concrete instructions (which words to cut, what to add, which sentences to split) to improve the summary.
Summary: {summary}
Report: {result["report"]}
Rules: {task.rules_text()}
Return ONLY the instructions. DO NOT rewrite the summary."""

    # Step 3 - one model call, same shape as before.
    #   temperature=0.3 is a good choice here: you want a careful reader,
    #   not a creative one.
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    # Step 4 - return the reply, stripped.
    return response.choices[0].message.content.strip()

    ### END CODE HERE ###

In [11]:
print(unittests.test_critique_summary(critique_summary)[1])

PASS  test_critique_summary


---

### Exercise 3 — `revise_summary`  *(30 points)*

**What the function does**: it is given the draft and the critique, and returns a
repaired summary.

**What the tests check**:

* exactly **one** call to the model
* the prompt contains the draft, **and** the critique, **and** the rules
* the function returns the model's reply word for word

**The trap in this one.** Whatever this function returns goes straight into
`check_summary()` and gets counted. If the model begins its reply with "Here is
the revised summary:", that is five extra words against the 100-word limit — and
in Part D you may watch a score go *down* after a revision purely because of it.

So tell the model to return the summary and nothing else: no heading, no preamble,
no list of the changes it made. Putting that instruction at the very end of your
prompt tends to work better than burying it in the middle.

In [12]:
def revise_summary(summary, critique, task, client=None, model=DRAFT_MODEL):
    """Rewrite `summary` so that it follows the critique and the rules."""
    client = client or get_client()

    ### START CODE HERE ###

    # Step 1 - build the prompt. It MUST contain all three:
    #       {summary}             - the current draft
    #       {critique}            - the editor's instructions
    #       {task.rules_text()}   - the rules again, so it knows what to aim at
    #   End the prompt with: return ONLY the revised summary, nothing else.
    prompt = f"""Rewrite the summary below so that every problem raised by the reviewer is fixed. Make sure to follow all the rules strictly.
Summary: {summary}
Critique: {critique}
Rules: {task.rules_text()}
Return ONLY the revised summary, nothing else."""

    # Step 2 - one model call.
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )

    # Step 3 - return the reply, stripped.
    return response.choices[0].message.content.strip()

    ### END CODE HERE ###

In [13]:
print(unittests.test_revise_summary(revise_summary)[1])

PASS  test_revise_summary


### All three tests at once

Everything above ran offline. When this cell prints `3/3`, the graded part of the
assignment is finished — and it has not used a single API call.

In [14]:
unittests.run_all_tests(generate_summary, critique_summary, revise_summary)

PASS  test_generate_summary
PASS  test_critique_summary
PASS  test_revise_summary

3/3 tests passed.


True

---

## Part C — Run it once, for real

*Not graded. This is where you get to look at what you built.*

Here is the payoff. Your three functions, chained together, are a reflection
agent, and it is five lines:

```python
draft    = generate_summary(task)                  # write it
before   = check_summary(draft, task)              # score it
critique = critique_summary(draft, task)           # find what is wrong
revised  = revise_summary(draft, critique, task)   # fix it
after    = check_summary(revised, task)            # score it again
```

Lines 1, 3 and 4 are the three functions you just wrote. Lines 2 and 5 are the
checker, there so you can see whether the middle actually helped.

This takes about twenty seconds, and most of that is deliberate waiting:
`get_client()` spaces its calls five seconds apart automatically, which keeps you
under the free tier's limit of 15 requests per minute.

In [ ]:
draft = generate_summary(PRACTICE_TASK, client=client)
before = check_summary(draft, PRACTICE_TASK)

critique = critique_summary(draft, PRACTICE_TASK, client=client)

revised = revise_summary(draft, critique, PRACTICE_TASK, client=client)
after = check_summary(revised, PRACTICE_TASK)

show_output(f"Draft — {before['score']}/4", draft, background="#fff8dc")
show_output("What the checker said about it", before["report"], background="#fce4ec")
show_output("Critique", critique, background="#e0f7fa")
show_output(
    f"Revised — {after['score']}/4",
    revised,
    background="#e8f5e9" if after["passed"] else "#fff8dc",
)
print(after["report"])

RULE CHECK REPORT
score: 4/4
  [PASS] length: 60 words (limit 100)
  [PASS] grounding: all required strings present
  [PASS] plain_language: no banned terms used
  [PASS] readability: longest sentence is within 20 words


**Did it reach 4/4?** Maybe, maybe not — and either way is fine, because none of
this is graded.

One pass of reflection fixes most of what the checker flagged, but not reliably
all of it. A revision that removes a banned word can easily introduce a 22-word
sentence in the process, and nothing gets to look a second time.

The obvious fix is to put the three steps in a loop: check, critique, revise,
check again, and keep going until the summary passes or you run out of budget.
That is where this pattern goes next in the course. You are welcome to try it now
if you are curious — it is not required and earns no extra points.

---

## Part D — Does reflection actually help?  *(not graded — just run it)*

Part C ran reflection on one abstract. A single result could be luck. This cell
runs your same three functions over **five different abstracts** and scores each
one twice:

* **One-shot** — the first draft, judged on its own. Costs 1 call to the model.
* **Reflection** — that same draft after one round of critique and revision.
  Costs 2 more calls, 3 in total.

The gap between those two scores is what this entire module is about, measured on
code you wrote: what did those 2 extra calls per summary actually buy?

Run the cell as it is; do not change it. It takes about a minute, and almost all of
that is the deliberate waiting between calls. **Whatever numbers come out are
fine** — high or low, they do not affect your grade.

In [ ]:
rows = []
for bench_task in BENCHMARK_TASKS:
    draft = generate_summary(bench_task, client=client)
    before = check_summary(draft, bench_task)

    critique = critique_summary(draft, bench_task, client=client)
    revised = revise_summary(draft, critique, bench_task, client=client)
    after = check_summary(revised, bench_task)

    rows.append(
        {
            "task": bench_task.id,
            "before": before,
            "after": after,
        }
    )
    print(
        f"{bench_task.id}: one-shot {before['score']}/4  ->  "
        f"reflection {after['score']}/4   (1 call  ->  3 calls)"
    )

n = len(rows)
one_total = sum(r["before"]["score"] for r in rows)
ref_total = sum(r["after"]["score"] for r in rows)

print()
print(f"one-shot   : {one_total}/{4 * n} points, {n} calls")
print(f"reflection : {ref_total}/{4 * n} points, {3 * n} calls")
print(f"delta      : {ref_total - one_total:+d} points for {2 * n:+d} calls")

b1: one-shot 3/4  ->  reflection 4/4   (1 call  ->  3 calls)
b2: one-shot 4/4  ->  reflection 4/4   (1 call  ->  3 calls)
b3: one-shot 4/4  ->  reflection 4/4   (1 call  ->  3 calls)
b4: one-shot 4/4  ->  reflection 4/4   (1 call  ->  3 calls)
b5: one-shot 4/4  ->  reflection 4/4   (1 call  ->  3 calls)

one-shot   : 19/20 points, 5 calls
reflection : 20/20 points, 15 calls
delta      : +1 points for +10 calls


The totals tell you *how much* reflection helped. The next cell tells you *where*:
which rules the first drafts broke, and which of those survived the revision.

Watch `length` and `readability` especially. A model cannot count the words in a
sentence it has not finished writing. Those two rules are close to guaranteed to
fail in a first draft, and they are exactly the failures a checker can catch and a
model cannot — which is the whole reason Exercise 2 made you hand the report to
the critic.

In [17]:
from collections import Counter

# Reuses the drafts from the cell above - no extra model calls.
broke_before = Counter()
still_broken = Counter()
for row in rows:
    for r in row["before"]["results"]:
        if not r["ok"]:
            broke_before[r["rule"]] += 1
    for r in row["after"]["results"]:
        if not r["ok"]:
            still_broken[r["rule"]] += 1

print(f"{'rule':16s} {'broken in first draft':>22s} {'still broken after':>20s}")
for rule in ["length", "grounding", "plain_language", "readability"]:
    print(f"{rule:16s} {broke_before[rule]:>22d} {still_broken[rule]:>20d}")

rule              broken in first draft   still broken after
length                                0                    0
grounding                             0                    0
plain_language                        0                    0
readability                           1                    0


---

## Submitting

1. **Restart the kernel and run every cell in order.** In Jupyter: Kernel →
   Restart & Run All. In VS Code: Restart, then Run All. A notebook whose cells
   were run out of order will not be graded.
2. Confirm the `run_all_tests` cell prints `3/3`. That is your grade.
3. Parts C and D should show real output, but their numbers do not affect your
   score.
4. Submit **only** the `.ipynb` file. Do not submit `.env` — it contains your
   personal API key — and do not submit `utils.py` or `unittests.py`; we have
   those.